Подключение диска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Подключение необходимых библиотек

In [ ]:
import random
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
from tqdm.notebook import tqdm

from torch import nn
from torchsummary import summary
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Создание класса для работы с датасетом

In [ ]:
import os
from PIL import Image
from typing import Tuple

class ImageDataset:
    def __init__(self, root_dir, transform=None):
        """
        Инициализация класса.

        :param root_dir: Путь к папке с подкаталогами, где каждый подкаталог представляет класс и содержит изображения.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = {}
        self.image_paths = []
        self.labels = []

        # Сканируем папку и создаём список изображений и их классов
        for class_idx, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                self.class_to_idx[class_name] = class_idx
                for image_name in os.listdir(class_path):
                    image_path = os.path.join(class_path, image_name)
                    self.image_paths.append(image_path)
                    self.labels.append(class_idx)

    def __len__(self) -> int:
        """Возвращает количество изображений в датасете."""
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, int]:
        """
        Возвращает изображение и номер его класса по индексу.

        :param idx: Индекс изображения.
        :return: Кортеж (изображение, номер класса).
        """
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

dir = '/content/drive/MyDrive/datasets/Accords/'

Архитектура модели

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        batch, channels, _, _ = x.size()
        y = self.global_pool(x).view(batch, channels)
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(batch, channels, 1, 1)
        return x * y


class SkipConnectionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SkipConnectionBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.dropout1 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout2 = nn.Dropout2d(0.3)  # Добавлено dropout

        self.se = SEBlock(out_channels)  # Добавляем SEBlock

        # Используем skip_conv только если количество каналов не совпадает
        self.use_skip_conv = in_channels != out_channels
        if self.use_skip_conv:
            self.skip_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        identity = self.skip_conv(x) if self.use_skip_conv else x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout1(out)  # Dropout после первого слоя
        out = self.bn2(self.conv2(out))
        out = self.dropout2(out)  # Dropout после второго слоя
        out = self.se(out)  # Применяем SEBlock
        out += identity
        return F.relu(out)


class MyModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            SkipConnectionBlock(3, 16),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(16, 64),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(64, 128),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(128, 256),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(256, 512),
            nn.MaxPool2d(2, 2),

            SkipConnectionBlock(512, 1024),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



model = MyModel(num_classes=6).to(device)
summary(model, (3, 180, 180))

Подготовка датасета, аугументация

In [ ]:
BATCH_SIZE = 16


transform_orgin = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform1 = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform2 = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform3 = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


full_dataset1 = ImageDataset(root_dir=dir, transform=transform1)
full_dataset2 = ImageDataset(root_dir=dir, transform=transform2)
full_dataset3 = ImageDataset(root_dir=dir, transform=transform3)
full_dataset_orgin = ImageDataset(root_dir=dir, transform=transform_orgin)

# Аугументация
combined_dataset = ConcatDataset([full_dataset_orgin, full_dataset1, full_dataset2, full_dataset3])
train_dataset, valid_dataset = random_split(combined_dataset, [0.9, 0.1])

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

len(combined_dataset)

Подготовка к обучению

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


train_losses = []
val_losses = []

train_acc_scores = []
val_acc_scores = []

best_val_acc = 0.0
best_model_path = '/content/drive/MyDrive/best_model1.pth' # Сохраняет модель на диск, что бы не потерять при закончившихся ресурсах колаба

num_epochs = 25

Цикл обучения

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_true = []
    train_pred = []

    for batch in tqdm(train_dataloader):
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_true.extend(labels.cpu().numpy())
        train_pred.extend(preds.cpu().numpy())

    train_acc = accuracy_score(train_true, train_pred)
    train_losses.append(running_loss / len(train_dataloader))
    train_acc_scores.append(train_acc)


    model.eval()
    val_running_loss = 0.0
    val_true = []
    val_pred = []

    # валидационный цикл, когда мы оцениваем качество работы модели на отложенной выборке
    with torch.no_grad():
        for batch in tqdm(valid_dataloader):
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_true.extend(labels.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = accuracy_score(val_true, val_pred)
    val_losses.append(val_running_loss / len(valid_dataloader))
    val_acc_scores.append(val_acc)

    # если получившаяся модель лучше предыдущей, сохраним чекпоинт
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'New best model saved with Accuracy: {best_val_acc:.4f}')


    # выведем в консоль получившиеся результаты на отдельной эпохе
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_losses[-1]:.4f}, Train Accuracy: {train_acc:.4f}, '
          f'Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {val_acc:.4f}')

Рисование Графиков

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc_scores, label='Train Accuracy')
plt.plot(val_acc_scores, label='Validation Accuracy')
plt.title('Accuracy Score')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Score')
plt.legend()

plt.show()